# B5.2 — Training/validation failure-slice audit

This notebook performs no training and never reads a test split. It reuses the authoritative B2/B4 checkpoints, exports path-aligned train/validation probabilities for both frozen development protocols, and audits layout family, metal density, model disagreement, and calibration. Exact geometry fields are reported as unavailable unless a separately generated edge-pair annotation JSONL is supplied.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
PROJECT_DRIVE = Path('/content/drive/MyDrive/ADVLSI2 2026 Project')
EXPERIMENTS_ROOT = PROJECT_DRIVE / 'experiments'
B2_CHECKPOINTS = EXPERIMENTS_ROOT / 'B2_baselines/b2_baselines/checkpoints'
B4_CHECKPOINTS = EXPERIMENTS_ROOT / 'B4_compact_architecture/b4_architecture/checkpoints'
OUTPUT_ROOT = EXPERIMENTS_ROOT / 'B5_2_failure_audit'
PREDICTIONS = OUTPUT_ROOT / 'predictions'
AUDIT_OUTPUT = OUTPUT_ROOT / 'failure_audit'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
assert B2_CHECKPOINTS.is_dir(), f'Missing B2 checkpoints: {B2_CHECKPOINTS}'
assert B4_CHECKPOINTS.is_dir(), f'Missing B4 checkpoints: {B4_CHECKPOINTS}'
print(f'Persistent output: {OUTPUT_ROOT}')

In [ ]:
import os
import shutil
import subprocess
import sys
import tempfile

REPOSITORY = 'https://github.com/nocleo/ADVLSI2_Project_updated.git'
BRANCH = 'agent/b5-failure-slice-audit'
CHECKOUT = Path(tempfile.mkdtemp(prefix='advlsi-b5-2-')) / 'ADVLSI2_Project_updated'
subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPOSITORY, str(CHECKOUT)], check=True)
os.chdir(CHECKOUT)

import torch
assert torch.cuda.is_available(), 'Select a GPU runtime for the probability export.'
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

In [ ]:
environment = os.environ.copy()
environment['PYTHONUNBUFFERED'] = '1'
command = [
    sys.executable, 'scripts/run_b5_failure_audit.py',
    '--manifest', 'data/b1_current_audit/manifest.json',
    '--dataset', 'training_datasets/combined_training_dataset.zip',
    '--predictions', str(PREDICTIONS),
    '--output-dir', str(AUDIT_OUTPUT),
    '--b2-checkpoints', str(B2_CHECKPOINTS),
    '--b4-checkpoints', str(B4_CHECKPOINTS),
    '--export-missing',
    '--device', 'cuda',
    '--batch-size', '256',
]
print(' '.join(command))
subprocess.run(command, check=True, env=environment)

In [ ]:
from IPython.display import Markdown, display
report = AUDIT_OUTPUT / 'README.md'
if report.exists():
    display(Markdown(report.read_text()))
else:
    print('No report was written; inspect the preceding error and rerun. Existing prediction exports will be reused.')

In [ ]:
from google.colab import files
archive = shutil.make_archive('/content/ADVLSI2_B5_2_results', 'zip', AUDIT_OUTPUT)
files.download(archive)